# Control Code For Perovskite PL-IV

In [1]:

# Import specific packages for using quantalux camera
import os
root_dir = 'C:\\Users\\FRG-users\\Documents\\GitHub\\Instruments\\FRG Hardware\\frghardware\\components\\frghardware\\components'
os.environ['PATH'] = os.path.join(root_dir, 'dlls', 'Native_64_lib') + os.pathsep + os.environ['PATH']

from thorlabs_tsi_sdk.tl_camera import TLCameraSDK, OPERATION_MODE
from thorlabs_tsi_sdk.tl_camera_enums import DATA_RATE


# import pickle
import numpy as np
import time
import cv2
import time
import h5py
import pandas as pd

import time
from pyvisa.errors import VisaIOError
from pyvisa.constants import VI_ERROR_TMO

%matplotlib notebook
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from tqdm import tqdm
import datetime
import winsound

# Psk stuff
#                                           1. Qtx camera
from frghardware.keithleyjv import control3 #2. Keithley Load
from frghardware.components.CNI import CNI #3. 635 nm CNI laser

# Si stuff
# from frghardware.components.FLIR import FLIR #4. FLIR InGaAs Camera
# from frghardware.components.kepco import Kepco #5. Kepco Load
from frghardware.components.laser import Laser808 #6. 808nm Jenoptik Laser

# Commonly used stuff
from frghardware.components.tec import omega #7. Omega PID Controller
from frghardware.components.opfo import opfo #8. Optics Focus XY Stage
from frghardware.components.daqPL3 import PLDAQ3 #9. Thorlabs Photodiode 
import u6 ##10 labjack U6 pro

# import pyvisa as visa
# from ThorlabsPM100 import ThorlabsPM100
# rm = visa.ResourceManager()
# inst = rm.open_resource('USB0::0x1313::0x8072::P2006516::INSTR', timeout=1)


today = datetime.datetime.now() # create a folder in Data that's today's date
path =  f"x{today.strftime('%Y%m%d')}" 
root = f"F:\\{path}"
if not os.path.exists(root):
    os.mkdir(root)

datafolder = os.path.join(root, 'Data')
if not os.path.exists(datafolder):
    os.mkdir(datafolder)


# Put quantalux here because I somehow can't import it from a .py file. Fine. ZJD 2024/04/02
class qtx:
    def __init__(self):        
        pass
    
    def capture(self, exposure_t, NUM_FRAMES, MODE = 'FPS_30'):
        with TLCameraSDK() as sdk:
            available_cameras = sdk.discover_available_cameras()
            if len(available_cameras) < 1:
                print("no cameras detected")

            with sdk.open_camera(available_cameras[0]) as camera:
                image_height = camera.image_height_pixels
                image_width = camera.image_width_pixels

                raw = np.zeros((image_height, image_width, NUM_FRAMES))

                if MODE == "FPS_30": #Low read noise mode
                    camera.data_rate = DATA_RATE.FPS_30
                if MODE == "FPS_50": #High frame rate mode
                    camera.data_rate = DATA_RATE.FPS_50
                camera.exposure_time_us = exposure_t  # set exposure to 11 ms
                camera.frames_per_trigger_zero_for_unlimited = 0  # start camera in continuous mode
                camera.image_poll_timeout_ms = 7100  # 5 second polling timeout, don't expect to ever need it

                camera.arm(2) #what is this?
                camera.issue_software_trigger() # begin aquisition

                try:
                    for i in tqdm(range(NUM_FRAMES), desc="Capturing frames", unit="frame"):
                        frame = camera.get_pending_frame_or_null()
                        if frame is not None:
    #                         print("frame #{} received!".format(frame.frame_count))

                            frame.image_buffer  # .../ perform operations using the data from image_buffer

                            #  NOTE: frame.image_buffer is a temporary memory buffer that may be overwritten during the next call
                            #        to get_pending_frame_or_null. The following line makes a deep copy of the image data:
                            image_buffer_copy = np.copy(frame.image_buffer)
                            raw[:, :, i] = np.squeeze(image_buffer_copy)
                            
#                             if np.any(raw[:, :, i] >= 65535):
#                                 raise ValueError("The img contains a number greater than or equal to 65536, quiting...")
#                                 break
                        else:
                            print("Unable to acquire image, program exiting...")
                            exit()
                except False:
                    print("Something's wrong, troubleshoot quantalux.py")
                print('disarming camera')
                camera.disarm()
#                 avg = np.mean(raw, axis = 2)
#                 std = np.std(raw, axis = 2)

        #  Because we are using the 'with' statement context-manager, disposal has been taken care of.

        return raw.astype(np.float16) #avg.astype(np.float16), std.astype(np.float16), 


    def stream(self): #Sometimes work. Just use ThorCam to live stream.

        with TLCameraSDK() as sdk:
            available_cameras = sdk.discover_available_cameras()
            if len(available_cameras) < 1:
                print("no cameras detected")

            with sdk.open_camera(available_cameras[0]) as camera:
                camera.exposure_time_us = 11000  # set exposure to 11 ms
                camera.frames_per_trigger_zero_for_unlimited = 0  # start camera in continuous mode
                camera.image_poll_timeout_ms = 1000  # 1 second polling timeout

                camera.arm(2) #what is this?
                camera.issue_software_trigger() # begin aquisition

                try:
                    while True:
                        frame = camera.get_pending_frame_or_null()
                        if frame is not None:
                            print("frame #{} received!".format(frame.frame_count))
                            frame.image_buffer
                            image_buffer_copy = np.copy(frame.image_buffer)
                            numpy_shaped_image = image_buffer_copy.reshape(camera.image_height_pixels, camera.image_width_pixels)
                            nd_image_array = np.full((camera.image_height_pixels, camera.image_width_pixels, 3), 0, dtype=np.uint8)
                            nd_image_array[:,:,0] = numpy_shaped_image
                            nd_image_array[:,:,1] = numpy_shaped_image
                            nd_image_array[:,:,2] = numpy_shaped_image

                            cv2.imshow("Press [enter] to close window", nd_image_array)

                            if cv2.waitKey(1) == 13:
                                print("loop terminated")
                                break

                        else:
                            print("Unable to acquire image, program exiting...")
                            exit()
                except False:
                    print("Something's wrong, troubleshoot quantalux.py")

                cv2.destroyAllWindows()
                camera.disarm()

        #  Because we are using the 'with' statement context-manager, disposal has been taken care of.

        print("streaming completed")    


    def binary_search_exposure_t(self):
        # Define exposure range
        self.low = 1000      # Minimum reasonable exposure time
        self.high = 7050000  # Maximum exposure time
        self.tolerance = 1000

        self.iteration_success = True  # Initialize success flag
        self.best_exposure = None      # Store optimal exposure time
        self.binary_search_NUM_FRAMES = 3

        while self.low <= self.high:
            exposure_t_bin_sear = (self.low + self.high) // 2  # Midpoint of binary search

            # Capture an image with the current exposure
            image = self.capture(exposure_t = exposure_t_bin_sear, NUM_FRAMES = self.binary_search_NUM_FRAMES)  # Ensure `capture()` is correctly implemented

            # Find the maximum pixel value
            max_pixel = np.max(image[:, :, 2][~np.isinf(image[:, :, 2])])

            if np.isinf(image[:, :, 2]).any():  # If any `inf` pixels exist, reduce exposure
                self.high = exposure_t_bin_sear - 1
                print(f"Reducing exposure: {exposure_t_bin_sear} us (too high)")

            elif max_pixel < 65536 - self.tolerance:  # If too low, increase exposure
                self.low = exposure_t_bin_sear + 1
                print(f"Increasing exposure: {exposure_t_bin_sear} us (too low)")
                self.best_exposure = exposure_t_bin_sear  # Store the latest valid exposure

            else:  # Pixel values are within the optimal range
                self.best_exposure = exposure_t_bin_sear
                print(f"Optimal exposure found: {exposure_t_bin_sear}")
                break

            time.sleep(0.1)  # Allow camera to adjust if needed

        else:
            print("Binary search ended. Best found exposure:", self.best_exposure)
            self.iteration_success = False
        return self.best_exposure

    
class control:
    # keithley allow 0-200 V, laser is the 635 nm CNI laser
    def __init__ (self):
        # hardware properties
        
        self.__laserON = False
        self.__keithleyON = False
        self.__cameraON = False
        
        # measurement settings
        self.bias = 0 #bias applied to sample
        self.laserpower = 0 #current supplied to laser ###may replace this with n_suns, if calibration is enabled
        self.vocmode = True # ZJD added 2024/05/26
        self.saturationtime = 0 # #current supplied to laser ###may replace this with n_suns, if calibration is enabled, not sure if i would have something like this
        self.numIV = 20 
        self.numframes = 50	#number of image frames to average
        self.__temperature = 25	#TEC stage temperature setpoint (C) during measurement
        self.temperatureTolerance = 0.2	#how close to the setpoint we need to be able to take a measurement (C)
        self.maxSoakTime = 60	# max soak time, in seconds, to wait for temperature to reach set point. If we reach this point, just go ahead with the measurement
        self.note = ''
#         self._spotMap = None	# optical power map of laser spot, used for PL normalization
        self._sampleOneSun = None # fractional laser power with which to approximate one-sun injection levels
        self._sampleOneSunJsc = None # target Jsc, matching of which is used for one-sun injection level is approximated
        self._sampleOneSunSweep = None # fractional laser power vs photocurrent (Isc), fit to provide one-sun estimate
        self.__previewFigure = None	#handle for matplotlib figure, used for previewing most recent image results
        self.__previewAxes = [None, None]	# handle for matplotib axes, used to hold the image and colorbar
        self.__backgroundImage = None

        self.outputDirectory = datafolder # changed by ZJD as compared to Rishi's code 2023/11/07
        self.sampleName = None
        self.__dataBuffer = [] # buffer to hold data files during sequential measurements of single sample. Held until a batch export
        
        # stage/positioning constatns for Psk solar cell side
        self.__sampleposition = (85, 165) # add in later #position where TEC stage is contered in camera FOV, um
        self.__detectorposition = (85, 105) #delta position between detector and sampleposition, um. The Thorlabs pd's position
        self.__fov = (120, 120) #dimensions of FOV, um

        # stage/positioning constatns for Si solar cell side
        
        # For PSK PLIV 
        self._scanSpeed_timestop = 1

        # PSK Camera parameters
        self.exposure_t = 30000 #in microseconds
        self.NUM_FRAMES = 200
        self.camera_MODE = 'FPS_30'

        # Params for saved load or laser
        self.load = None
        self.laser = None
        self.camera = None

        # stage constants for controlling opfo
        self._screw_pitch = 5 #mm
        self._stepper_angle = 1.8 #degree
        self._subdivision = 2
        self._pulse_equivalent = self._screw_pitch * self._stepper_angle / (360 * self._subdivision)
        self._actual_displacement = 23 #mm, user input here to set movement distance
        self._pulse_number = self._actual_displacement / self._pulse_equivalent #input this to the axis

        speed_value = 50 #user input here to change movement speed
        self._actual_speed = (speed_value + 1) * 22000 * self._pulse_equivalent / 720

        self.connect()
        
    @property
    def temperature(self):
            return self.__temperature

    @temperature.setter
    def temperature(self, t):
        self.tec.setpoint = t
        self.__temperature = t

    def connect(self):
        # Psk stuff
        self.qtx = qtx() # connect to qtx camera
        self.keithley = control3.Control(address = 'GPIB1::20::INSTR') # connect to Keithley 2401

        self.cni = CNI() # Connect to CNI laser
        
        # Si stuff
#         self.flir = FLIR()# connect to FLIR camera
#         self.kepco = Kepco()# connect to Kepco
# #         self.kepco.set(voltage=0)   # set voltage to 0, seems to solve current compliance issues
#         self.jenoptik = Laser808()
  
        # Commonly used stuff
        self.daq3 = PLDAQ3() # connect to NI-USB6000 DAQ
        self.opfo = opfo() # connect to FRG stage
        self.tec = omega() # connect to omega PID controller, which is driving the TEC stage.
        
#         self.power_meter = ThorlabsPM100(inst=inst) #For intensity mapping
#         self.lju = u6.U6() # For subcell V measurement
#         self.lju.getCalibrationData()
        
    
    def disconnect(self): 
        
        print('qtx is disconencted as long as it is not taking images or streaming')
        
        try:
            self.keithley.disconnect()
        except:
            print('Could not disconnect Keithley load')

        try:
            self.cni.disconnect()
        except:
            print('Could not disconnect CNI laser')
            
            
#         try:
#             self.flir.disconnect()
#         except:
#             print('Could not disconnect FLIR camera')
#         try:
#             self.kepco.disconnect()
#         except:
#             print('Could not disconnect Kepco')            
#         try:
#             self.jenoptik.disconnect()
#         except:
#             print('Could not disconnect jenoptik/OSTech Laser')           
            
        try: 
            self.daq3.disconnect()
        except:
            print('Could not disconnect DAQ')
        try:
            self.opfo.disconnect()
        except:
            print('Could not disconnect opfo stage')
        try:
            self.tec.disconnect()
        except: 
            print('Could not disconnect TEC controller')

#     def _LJ_getV_50x(self, high = None, GND = 12):
#         test_folder = []

#         for _ in range(50):
#             num = self.lju.getAIN(high) - self.lju.getAIN(GND)

#             test_folder.append(num)
#         return np.nanmean(test_folder)
    
#     def measureSUBCELL(self, subcellnum):
#         """
#         Measure voltage difference between specific channels
#         depending on the subcell number.
#         """

#         # Define normal mapping
#         mapping = {
#             7:  (1, 3),
#             6:  (3, 5),
#             5:  (5, 7),
#             4:  (7, 9),
#             3: (9, 11),
#             2: (11, 13)
#         }

#         # Special case: second 7 (or “reverse mode”)
#         if subcellnum == 1:  # you can choose any special code you like
#             pos, neg = 13, 12
#         else:
#             if subcellnum not in mapping:
#                 raise ValueError(f"Unsupported subcell number: {subcellnum}")
#             pos, neg = mapping[subcellnum]

#         return self._LJ_getV_50x(pos) - self._LJ_getV_50x(neg)


    ### helper functions
    def _waitForTemperature(self):
        refreshDelay = 0.5	#how long to wait between temperautre checks, in seconds
        reachedTemp = False

        startTime = time.time()
        while (not reachedTemp) and (time.time() - startTime <= self.maxSoakTime):
            currentTemp = self.tec.getTemperature()
            if np.abs(currentTemp - self.temperature) <= self.temperatureTolerance:
                reachedTemp = True
            else:
                time.sleep(refreshDelay)

        if not reachedTemp:
            print('Did not reach {0} C within {1} seconds: starting measurement anyways.'.format(self.temperature, self.maxSoakTime))

        return True
    def _getOpticalPower(self):
        '''
        reads signal from photodetector, converts to optical power using calibration vs thorlabs Si power meter (last checked 2019-08-20)
        calibrationFit = [-0.1145, 9.1180]; #polyfit of detector reading vs (Si power meter / detector reading), 2019-08-20
        '''

        voltage, _, _ = self.daq3.acquire()
        power = voltage/1000/0.311/0.13*1000 / 54 
        #Pd readout / Resistor / Reponsivity / pd active area in cm2 * (1000mW/W), ZJD 2023/11/13
        
        #* (calibrationFit[0]*voltage + calibrationFit[1])	#measured optical power, units of mW/cm^2

        return power

    def _backgroundCorrection(self, img):
        img = img - self.__backgroundImage
        img[img<0] = 0

        return img
    
    def _laserIntensityCorrection(self, array):
        '''
        Created this function to avoid laser current range 248～255, 503～511, 758～767
        Because this laser controller has problems (defect from device). Sad T^T
        '''
        # Multiply the array by x and then by 1000
        CNI_input = array * 1000

        # Update values in the specified ranges
        CNI_input[(CNI_input >= 248) & (CNI_input <= 255)] = 256
        CNI_input[(CNI_input >= 503) & (CNI_input <= 511)] = 512
        CNI_input[(CNI_input >= 758) & (CNI_input <= 767)] = 768
        
        # Return the new array
        array = CNI_input / 1000
        
        return array

    def _intensity_mapping_snaking(self, stoptime_ratio = 1.3): 
        '''
        Created this function to do intensity mapping using opfo.  
        ZJD 2024/03/28
        '''
        target_x_coor, target_y_coor = self.opfo.target_x_coor, self.opfo.target_y_coor
        pixel_pitch = self.opfo.pixel_pitch

        x_mappin_diameter, y_mappin_diameter = self.opfo.x_mappin_diameter, self.opfo.y_mappin_diameter
        map_name = self.opfo.map_name

        x_map_half_radius, y_map_half_radius = x_mappin_diameter/2, y_mappin_diameter/2

        #move to bottom left:
        x_start = target_x_coor - x_map_half_radius
        y_start = target_y_coor + y_map_half_radius

        self.opfo.xmove_to_coor(x_start)
        self.opfo.ymove_to_coor(y_start)

        ##generate an empty grid
        x_grid = int(x_mappin_diameter/pixel_pitch)
        y_grid = int(y_mappin_diameter/pixel_pitch)
        matrix = np.full((x_grid,y_grid, 5), np.nan) 

        flip = 1 #fwd
        x_position_tracker = x_start
        y_position_tracker = y_start

        for y_ in range(y_grid):
            for x_ in range(x_grid):
                if x_ != x_grid-1:
                    avg, std, data = self.daq3.acquire()
                    matrix[x_, y_, 0], matrix[x_, y_, 1], matrix[x_, y_, 2], matrix[x_, y_, 3], matrix[x_, y_, 4] = x_position_tracker, y_position_tracker, x_, y_, avg

                    if flip == 1:

                        pitch_to_move = pixel_pitch

                    if flip == -1:

                        pitch_to_move = -pixel_pitch

                    self.opfo.xmove(pitch_to_move)

                    slptime = pixel_pitch/self._actual_speed * stoptime_ratio
                    time.sleep(slptime)

                    x_position_tracker = x_position_tracker + pitch_to_move
                else:
                    avg, std, data = self.daq3.acquire()
                    matrix[x_, y_, 0], matrix[x_, y_, 1], matrix[x_, y_, 2], matrix[x_, y_, 3], matrix[x_, y_, 4] = x_position_tracker, y_position_tracker, x_, y_, avg
            if pixel_pitch < 10:
                time.sleep(0.8)
            self.opfo.ymove(-pixel_pitch)
            slptime = pixel_pitch/self._actual_speed * stoptime_ratio

            avg, std, data = self.daq3.acquire()
            time.sleep(slptime)

            y_position_tracker = y_position_tracker - pixel_pitch

            flip = flip * -1 #flip snaking direction
        np.save(f'{os.path.join(datafolder, map_name)}.npy', matrix)
        print(f'saved to {os.path.join(datafolder, map_name)}.npy')

        return matrix


#     def _intensity_mapping_snaking_power_meter(self, stoptime_ratio = 1.3): 
#         '''
#         Created this function to do intensity mapping using opfo. Data taken on a power meter instead of a photodiode.
#         ZJD 2024/03/28
#         '''
#         target_x_coor, target_y_coor = self.opfo.target_x_coor, self.opfo.target_y_coor
#         pixel_pitch = self.opfo.pixel_pitch

#         x_mappin_diameter, y_mappin_diameter = self.opfo.x_mappin_diameter, self.opfo.y_mappin_diameter
#         map_name = self.opfo.map_name

#         x_map_half_radius, y_map_half_radius = x_mappin_diameter/2, y_mappin_diameter/2

#         #move to bottom left:
#         x_start = target_x_coor - x_map_half_radius
#         y_start = target_y_coor + y_map_half_radius

#         self.opfo.xmove_to_coor(x_start)
#         self.opfo.ymove_to_coor(y_start)

#         ##generate an empty grid
#         x_grid = int(x_mappin_diameter/pixel_pitch)
#         y_grid = int(y_mappin_diameter/pixel_pitch)
#         matrix = np.full((x_grid,y_grid, 5), np.nan) 

#         flip = 1 #fwd
#         x_position_tracker = x_start
#         y_position_tracker = y_start

#         for y_ in range(y_grid):
#             for x_ in range(x_grid):
#                 if x_ != x_grid-1:
#                     avg, std, data = self.power_meter.read, np.nan, np.nan
#                     matrix[x_, y_, 0], matrix[x_, y_, 1], matrix[x_, y_, 2], matrix[x_, y_, 3], matrix[x_, y_, 4] = x_position_tracker, y_position_tracker, x_, y_, avg

#                     if flip == 1:

#                         pitch_to_move = pixel_pitch

#                     if flip == -1:

#                         pitch_to_move = -pixel_pitch

#                     self.opfo.xmove(pitch_to_move)

#                     slptime = pixel_pitch/self._actual_speed * stoptime_ratio
#                     time.sleep(slptime)

#                     x_position_tracker = x_position_tracker + pitch_to_move
#                 else:
#                     avg, std, data = self.power_meter.read, np.nan, np.nan
#                     matrix[x_, y_, 0], matrix[x_, y_, 1], matrix[x_, y_, 2], matrix[x_, y_, 3], matrix[x_, y_, 4] = x_position_tracker, y_position_tracker, x_, y_, avg
#             if pixel_pitch < 10:
#                 time.sleep(0.8)
#             self.opfo.ymove(-pixel_pitch)
#             slptime = pixel_pitch/self._actual_speed * stoptime_ratio

#             avg, std, data = self.power_meter.read, np.nan, np.nan
#             time.sleep(slptime)

#             y_position_tracker = y_position_tracker - pixel_pitch

#             flip = flip * -1 #flip snaking direction
#         np.save(f'{os.path.join(datafolder, map_name)}.npy', matrix)
#         print(f'saved to {os.path.join(datafolder, map_name)}.npy')

#         return matrix


    
c = control()

CNI connected
DAQ3 connected
OpFo connected
TEC controller connected


# Timeseries Test

# SunsPL/SunsVoc/JscVoc all together

In [18]:
# Set fraction of laser current. Max = 1, i.e. "1p00"

lscurr = ['1p00', '0p85','0p80','0p75', '0p70', '0p65', '0p60', '0p55', '0p50', '0p45', '0p40', '0p35', '0p30']

# folder number and scan number. Folder number increase as suns changes, scan number increase as more JV are scanned through the test
folder_num, snum = 0, 0

path = 'x20251113'
cell_num = 'xA15F55584'

SunsVocSunsPLJscVoc_folder = f'H:\\{path}_1sun_module\\Data\\SunsVocSunsPLJscVoc'
JVdatafolder = f'H:\\{path}_1sun_module\\Data\\JV'
if not os.path.exists(SunsVocSunsPLJscVoc_folder):
    os.mkdir(SunsVocSunsPLJscVoc_folder)
if not os.path.exists(JVdatafolder):
    os.mkdir(JVdatafolder)

In [ ]:

def take_JV(JVdatafolder, cell_num, snum, volmin = -1, volmax = xxxxx):# Set your own volmax

    c.cni.set(laser_curr = 1)
    c.cni.on()
    
    wait_time = 300  # seconds 

    for _ in tqdm(range(wait_time), desc= f"JV: Waiting on laser stability, {cell_num}_M1_S{snum}", unit="s"):
        time.sleep(1) #
        

    jv_name = os.path.join(JVdatafolder, f'{cell_num}_M1_S{snum}')

    c.keithley.area = xxxxx # Set your own cell area
    c.keithley.jv(jv_name, 'fwdrev', volmin, volmax, light = True)
    c.cni.off()
    print(f'JV: {cell_num}_M1_S{snum} done')



def keithley_off_twice(delay=0.5):
    try:
        c.keithley.off()
        time.sleep(delay)
        c.keithley.off()
    except Exception as e:
        print(f"Warning: failed while forcing Keithley off: {e}")

def take_JV(JVdatafolder, cell_num, snum, volmin=-1, volmax=xxxxx, max_retry=2):

    c.cni.set(laser_curr=xxxxx)
    c.cni.on()

    wait_time = 300  # seconds
    for _ in tqdm(range(wait_time),
                  desc=f"JV: Waiting on laser stability, {cell_num}_M1_S{snum}",
                  unit="s"):
        time.sleep(1)

    jv_name = os.path.join(JVdatafolder, f'{cell_num}_M1_S{snum}')
    c.keithley.area = xxxxx

    attempt = 0
    while attempt <= max_retry:
        try:
            c.keithley.jv(jv_name, 'fwdrev', volmin, volmax, light=True)
            break  # ✅ success → exit retry loop

        except VisaIOError as e:
            if e.error_code == VI_ERROR_TMO:
                attempt += 1
                print(f"⚠️ Keithley timeout during JV (attempt {attempt}/{max_retry})")

                keithley_off_twice(delay=0.5)
                time.sleep(30)  # let VISA settle

                if attempt > max_retry:
                    print("❌ JV retry limit exceeded")
                    raise
                else:
                    print("🔁 Retaking JV...")

            else:
                raise  # non-timeout VISA error → fail fast

    c.cni.off()
    print(f'JV: {cell_num}_M1_S{snum} done')


try: 
    for ls in lscurr:

    ################################## voc image ##################################
        c.keithley.voc()

    #         time.sleep(300)

        folder_num += 1
        laser_curr_level = ls
        counter = 0
        laser_curr = float(laser_curr_level.replace('p', '.'))


        voc_folder = []
        
        tmp_folder = []
        intensity_rawvoltage_folder = []
        laser_curr_folder = []
        exp_t_folder = []

        c.NUM_FRAMES = 200  

        voc_imgs_path = f'{SunsVocSunsPLJscVoc_folder}\\{path}_SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_voc_folder{folder_num}'
        if not os.path.exists(voc_imgs_path):
            os.mkdir(voc_imgs_path)

        counter += 1
        print(f'{counter} out of 8 imgs')

        im_bkg = c.qtx.capture(exposure_t = bkg_expt, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(voc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_voc_expt_{c.exposure_t}us_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

        voc = c.keithley.voc()
        voc_folder.append(voc)

        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        time.sleep(1)

        c.cni.set(laser_curr = laser_curr)
        c.cni.on()
        time.sleep(1)

        # Search for the exposure time after stead state has been reached
        wait_time = 300  # seconds 

        for _ in tqdm(range(wait_time), desc="Waiting on laser stabilization", unit="s"):
            time.sleep(1) #

        expt_searched = c.qtx.binary_search_exposure_t() #

        # Total duration in minutes and interval in minutes
        total_duration = 50 # min 
        interval = 5 # min 
        
        iterations = total_duration//interval
        
        expt_searched = int(0.9*expt_searched) # some pixels lightsoak during the wait time
        c.exposure_t = (expt_searched)


        for _ in range(int(iterations)):
            counter += 1
            print(f'{counter} out of {iterations + 2} imgs')
            im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
            np.save(os.path.join(voc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_voc_expt_{c.exposure_t}us_raw{counter}.npy'), im_bkg)

            tmp = c.tec.getTemperature()
            tmp_folder.append(tmp)

            voc = c.keithley.voc()
            voc_folder.append(voc)

            a,_,_ = c.daq3.acquire()
            intensity_rawvoltage_folder.append(a)

            laser_curr_folder.append(laser_curr)
            exp_t_folder.append(c.exposure_t)

            for _ in tqdm(range(interval*60), desc="Waiting on PSM stabilization", unit="s"): # 
                time.sleep(1) #

        c.cni.off()

        counter += 1
        print(f'{counter} out of {iterations + 2} imgs')

        # use the steady state exposure t to take another bkg image

        im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(voc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_voc_expt_{c.exposure_t}us_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

        voc = c.keithley.voc()
        voc_folder.append(voc)
        time.sleep(0.1)


        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        df = pd.DataFrame({'laser_curr': laser_curr_folder, 
                           'Voc (V)': voc_folder,                            
                           'tmp (C)': tmp_folder, 
                           'exp_t (us)': exp_t_folder, 
                           'intensity_rawvoltage (470ohm V)': intensity_rawvoltage_folder})
        df.to_excel(os.path.join(voc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_voc_expt_{c.exposure_t}us_output.xlsx'), index=False)


#         ################################## JV after voc img ##################################

        wait_time = 60  # seconds 

        for _ in tqdm(range(wait_time), desc="System interval: Waiting", unit="s"):
            time.sleep(1) #

        snum += 1

        take_JV(JVdatafolder, cell_num, snum)

        ################################# jsc image ##################################
        c.keithley.jsc()

        wait_time = 60  # seconds 

        for _ in tqdm(range(wait_time), desc="System interval: Waiting", unit="s"):
            time.sleep(1) #

        counter = 0

        jsc_folder = []
        tmp_folder = []
        intensity_rawvoltage_folder = []
        laser_curr_folder = []
        exp_t_folder = []
        notes_folder = []


        jsc_imgs_path = f'{SunsVocSunsPLJscVoc_folder}\\{path}_SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_jsc_folder{folder_num}'
        if not os.path.exists(jsc_imgs_path):
            os.mkdir(jsc_imgs_path)

        c.keithley.set_voltage(voltage = 0)
        c.keithley.on()

        counter += 1
        print(f'{counter} out of 8 imgs')

        im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(jsc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_jsc_expt_{c.exposure_t}us_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

        # voc = c.keithley.voc()
        # voc_folder.append(voc)
        _, jsc = c.keithley.read()
        jsc_folder.append(jsc)

        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        notes_folder.append('dark_initial')

        time.sleep(1)

        c.cni.set(laser_curr = laser_curr)
        c.cni.on()

        wait_time = 300  # seconds 

        for _ in tqdm(range(wait_time), desc="System interval: Waiting", unit="s"):
            time.sleep(1) #

        for _ in range(int(iterations)):
            counter += 1
            print(f'{counter} out of {iterations + 2} imgs')
            im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
            np.save(os.path.join(jsc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_jsc_expt_{c.exposure_t}us_raw{counter}.npy'), im_bkg)

            tmp = c.tec.getTemperature()
            tmp_folder.append(tmp)

        #     voc = c.keithley.voc()
        #     voc_folder.append(voc)
            _, jsc = c.keithley.read()
            jsc_folder.append(jsc)

            a,_,_ = c.daq3.acquire()
            intensity_rawvoltage_folder.append(a)

            laser_curr_folder.append(laser_curr)
            exp_t_folder.append(c.exposure_t)

            notes_folder.append('time series')

            time.sleep(interval * 60)  # Convert minutes to seconds # 

        c.cni.off()

        counter += 1
        print(f'{counter} out of {iterations + 2} imgs')

        im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(jsc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_jsc_expt_{c.exposure_t}us_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

        # voc = c.keithley.voc()
        # voc_folder.append(voc)
        _, jsc = c.keithley.read()
        jsc_folder.append(jsc)

        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        notes_folder.append('dark_end')

        c.keithley.off()

        df = pd.DataFrame({'laser_curr': laser_curr_folder, 'Jsc (A)': jsc_folder, 'tmp (C)': tmp_folder, 'exp_t (us)': exp_t_folder, 'intensity_rawvoltage (2000ohm V)': intensity_rawvoltage_folder, 'notes': notes_folder})
        df.to_excel(os.path.join(jsc_imgs_path, f'SunsVocSunsPLJscVoc_laser_curr{laser_curr_level}_jsc_expt_{c.exposure_t}us_output.xlsx'), index=False)

        ################################## JV after jsc img ##################################

        wait_time = 60  # seconds 

        for _ in tqdm(range(wait_time), desc="System interval: Waiting", unit="s"):
            time.sleep(1) #

        snum += 1

        take_JV(JVdatafolder, cell_num, snum)
        
except Exception as e:
    
    print(f"An error occurred: {e}")
    
    c.cni.off()
    c.keithley.off()

    duration = 5000  # milliseconds
    freq = 440  # Hz
    winsound.Beep(freq, duration)

# PL-v time series test

## PLv folder

In [2]:
vapplied_level_ss = ['1p1500', '1p1000','0p9500', '0p9000', '0p8500', '0p8250', '0p8000', '0p7750', '0p7500', '0p7250', '0p7000', '0p0000']

cell_num = 'xA15F55585'

path = 'x20260127'

PLv_folder = f'H:\\{path}_1sun_FULLmodule\\Data\\PLv'
JVdatafolder = f'H:\\{path}_1sun_FULLmodule\\Data\\JV'
if not os.path.exists(PLv_folder):
    os.mkdir(PLv_folder)
if not os.path.exists(JVdatafolder):
    os.mkdir(JVdatafolder)
folder_num, snum = 0, 0

In [ ]:
# c.cni.on()
# wait_time = 600  # seconds a

# for _ in tqdm(range(wait_time), desc= f"JV: Waiting on laser stability, {cell_num}_M1_S{snum}", unit="s"):
#     time.sleep(1) #
# c.cni.off()


def keithley_off_twice(delay=0.5):
    try:
        c.keithley.off()
        time.sleep(delay)
        c.keithley.off()
    except Exception as e:
        print(f"Warning: failed while forcing Keithley off: {e}")

def take_JV(JVdatafolder, cell_num, snum, volmin=-1, volmax=xxxxx, max_retry=2):

    c.cni.set(laser_curr=xxxxx)
    c.cni.on()

    wait_time = 300  # seconds
    for _ in tqdm(range(wait_time),
                  desc=f"JV: Waiting on laser stability, {cell_num}_M1_S{snum}",
                  unit="s"):
        time.sleep(1)

    jv_name = os.path.join(JVdatafolder, f'{cell_num}_M1_S{snum}')
    c.keithley.area = xxxxx

    attempt = 0
    while attempt <= max_retry:
        try:
            c.keithley.jv(jv_name, 'fwdrev', volmin, volmax, light=True)
            break  # ✅ success → exit retry loop

        except VisaIOError as e:
            if e.error_code == VI_ERROR_TMO:
                attempt += 1
                print(f"⚠️ Keithley timeout during JV (attempt {attempt}/{max_retry})")

                keithley_off_twice(delay=0.5)
                time.sleep(30)  # let VISA settle

                if attempt > max_retry:
                    print("❌ JV retry limit exceeded")
                    raise
                else:
                    print("🔁 Retaking JV...")

            else:
                raise  # non-timeout VISA error → fail fast

    c.cni.off()
    print(f'JV: {cell_num}_M1_S{snum} done')


try:
# time.sleep(300)
    for vapplied_level in vapplied_level_ss:
        folder_num += 1
        laser_curr_level = '0p29'
        laser_curr = float(laser_curr_level.replace('p', '.'))
        vapplied_level = vapplied_level
        vapplied = float(vapplied_level.replace('p', '.'))

#         c.exposure_t = 200000
        bkg_expt = 14000
        counter = 0
        jmeas_folder = []
        vmeas_folder = []
        
        tmp_folder = []
        intensity_rawvoltage_folder = []
        laser_curr_folder = []
        exp_t_folder = []
        notes_folder = []

        c.NUM_FRAMES = 200 

        plv_imgs_path = f'{PLv_folder}\\{path}_PLv_laser_curr{laser_curr_level}_plv_folder{folder_num}'
        if not os.path.exists(plv_imgs_path):
            os.mkdir(plv_imgs_path)

        c.keithley.voc()

        im_bkg = c.qtx.capture(exposure_t = bkg_expt, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(plv_imgs_path, f'PLv_laser_curr{laser_curr_level}_plv_expt_{c.exposure_t}us_vappled{vapplied_level}_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

#         voc = c.keithley.voc()
#         voc_folder.append(voc)

        vmeas_folder.append('dark_initial')
        jmeas_folder.append('dark_initial')

        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        notes_folder.append('dark_initial')

        time.sleep(1)

        c.cni.set(laser_curr = laser_curr)
        c.cni.on()
        time.sleep(1)

        # Turn on keithley
        c.keithley.set_voltage(voltage = vapplied)
        c.keithley.on()
        print(f'Vapplied: {vapplied}')

        wait_time = 300  # seconds 

        for _ in tqdm(range(wait_time), desc="PLv: Waiting on 1 sun laser stability", unit="s"):
            time.sleep(1) #

#         # Calculate the number of iterations
#         iterations = total_duration // interval
        
#         # Search for the exposure time after stead state has been reached
#         wait_time = 300  # seconds 

#         for _ in tqdm(range(wait_time), desc="Waiting on laser stabilization", unit="s"):
#             time.sleep(1) #
            
        expt_searched = c.qtx.binary_search_exposure_t() 
        
        # Total duration in minutes and interval in minutes
        total_duration = 50 # min 
        interval = 5 # min 
        
        if expt_searched >= 7000000:
            iterations = 4
        elif expt_searched < 7000000:
            iterations = total_duration // interval
    
        counter += 1
        print(f'{counter} out of {iterations + 2} imgs') # + 2 to include initial and end dark image
            
        expt_searched = int(0.9*expt_searched)
        c.exposure_t = (expt_searched)
        

        for _ in range(int(iterations)):
            counter += 1
            print(f'{counter} out of 8 imgs')
            im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
            np.save(os.path.join(plv_imgs_path, f'PLv_laser_curr{laser_curr_level}_plv_expt_{c.exposure_t}us_vappled{vapplied_level}_raw{counter}.npy'), im_bkg)

            tmp = c.tec.getTemperature()
            tmp_folder.append(tmp)

#             voc = c.keithley.voc()
#             voc_folder.append(voc) 
            vmeas, jmeas = c.keithley.read()
            vmeas_folder.append(vmeas)
            jmeas_folder.append(jmeas)      

            a,_,_ = c.daq3.acquire()
            intensity_rawvoltage_folder.append(a)

            laser_curr_folder.append(laser_curr)
            exp_t_folder.append(c.exposure_t)

            notes_folder.append('time series')

            time.sleep(interval * 60)  # Convert minutes to seconds 

        c.cni.off()
        c.keithley.off()

        counter += 1
        print(f'{counter} out of 8 imgs')

        im_bkg = c.qtx.capture(exposure_t = c.exposure_t, NUM_FRAMES = c.NUM_FRAMES, MODE = c.camera_MODE)
        np.save(os.path.join(plv_imgs_path, f'PLv_laser_curr{laser_curr_level}_plv_expt_{c.exposure_t}us_vappled{vapplied_level}_bkg{counter}.npy'), im_bkg)

        tmp = c.tec.getTemperature()
        tmp_folder.append(tmp)

#         voc = c.keithley.voc()
#         voc_folder.append(voc)
        # vmeas, jmeas = c.keithley.read()
        vmeas_folder.append('dark_end')
        jmeas_folder.append('dark_end')

        a,_,_ = c.daq3.acquire()
        intensity_rawvoltage_folder.append(a)

        laser_curr_folder.append(laser_curr)
        exp_t_folder.append(c.exposure_t)

        notes_folder.append('dark_end')

        # c.keithley.off()

        df = pd.DataFrame({'laser_curr': laser_curr_folder, 
                           'Jmeas (A)': jmeas_folder, 
                           'Vmeas (V)': vmeas_folder, 
                           'tmp (C)': tmp_folder, 
                           'exp_t (us)': exp_t_folder, 
                           'intensity_rawvoltage (2000ohm V)': intensity_rawvoltage_folder, 
                           'notes': notes_folder})
        df.to_excel(os.path.join(plv_imgs_path, f'PLv_laser_curr{laser_curr_level}_plv_vappled{vapplied_level}_expt_{c.exposure_t}us_output.xlsx'), index=False)

        ################################## JV after voc img ##################################

        wait_time = 60  # seconds 

        for _ in tqdm(range(wait_time), desc="System interval: Waiting", unit="s"):
            time.sleep(1) #

        snum += 1

        take_JV(JVdatafolder, cell_num, snum)

        
except Exception as e:
    
    print(f"An error occurred: {e}")
    
    c.cni.off()
    c.keithley.off()

    duration = 60000  # milliseconds
    freq = 440  # Hz
    winsound.Beep(freq, duration)